# Phase 1 Sensitivity Analysis

Tests whether Phase 1 findings are robust across VPIN parameter choices.

**Parameter grid:**
- `bucket_size`: {50, 100, 200, 500}
- `lookback`: {5, 10, 20, 50}
- Insider window: {last 10%, 20%, 30%} of volume buckets

**Robustness criterion:** Result must hold across >= 3 adjacent parameter settings.
If it only works at one exact setting, it's fragile and unreliable.

In [ ]:
import sys
from pathlib import Path

_notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
_repo_root = str(_notebook_dir.parent) if _notebook_dir.name == "notebooks" else str(_notebook_dir)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from IPython.display import display

from src.analysis.util.vpin import vpin_cte
from src.analysis.util.insider_cases import CASES, all_cases
from src.analysis.util.stats import ks_test

rng = np.random.default_rng(seed=42)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ── Parameters ──
BUCKET_SIZES = [50, 100, 200, 500]
LOOKBACKS = [5, 10, 20, 50]
WINDOW_PCTS = [0.10, 0.20, 0.30]

REPO_ROOT = Path(_repo_root)
CACHE_DIR = REPO_ROOT / "data" / "case_study" / "phase1"

print(f"Parameter grid: {len(BUCKET_SIZES)} × {len(LOOKBACKS)} × {len(WINDOW_PCTS)} = "
      f"{len(BUCKET_SIZES) * len(LOOKBACKS) * len(WINDOW_PCTS)} configurations per case")

In [ ]:
# ── Load cached normalized trades ──
case_trades = {}
con = duckdb.connect()

for case in all_cases():
    cache_path = CACHE_DIR / f"{case.case_id}_normalized_trades.parquet"
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        case_trades[case.case_id] = df
        print(f"  {case.case_id}: {len(df):,} trades")
    else:
        print(f"  {case.case_id}: SKIPPED — no cached data (run phase1_validation first)")

print(f"\nCases loaded: {len(case_trades)}")

In [ ]:
# ── Run parameter grid ──
grid_results = []  # List of dicts

total_configs = len(BUCKET_SIZES) * len(LOOKBACKS) * len(WINDOW_PCTS) * len(case_trades)
completed = 0

for case_id, trades_df in case_trades.items():
    table_name = f"sens_{case_id}"
    con.register(table_name, trades_df)

    for bucket_size, lookback, window_pct in product(BUCKET_SIZES, LOOKBACKS, WINDOW_PCTS):
        try:
            vpin_df = con.execute(f"""
                WITH {vpin_cte(table_name, bucket_size, lookback)}
                SELECT vpin FROM vpin_series
                WHERE window_size = {lookback}
                ORDER BY bucket_id
            """).df()

            if len(vpin_df) < 20:
                continue

            # Define insider window
            n = len(vpin_df)
            cutoff = int(n * (1 - window_pct))
            insider = vpin_df["vpin"].iloc[cutoff:].values
            non_insider = vpin_df["vpin"].iloc[:cutoff].values

            if len(insider) < 5 or len(non_insider) < 5:
                continue

            ks = ks_test(insider, non_insider, alternative="less")

            grid_results.append({
                "case_id": case_id,
                "bucket_size": bucket_size,
                "lookback": lookback,
                "window_pct": window_pct,
                "ks_d": ks.statistic,
                "ks_p": ks.pvalue,
                "significant": ks.significant,
                "insider_mean": np.mean(insider),
                "non_insider_mean": np.mean(non_insider),
                "n_buckets": n,
            })
        except Exception as e:
            pass

        completed += 1
        if completed % 50 == 0:
            print(f"  {completed}/{total_configs} configurations completed...")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search complete: {len(grid_df)} valid configurations")
print(f"Significant (p<0.05): {grid_df['significant'].sum()} ({grid_df['significant'].mean():.0%})")

In [ ]:
# ── Heatmaps: KS D across bucket_size × lookback, per case ──
# Fix window_pct = 0.20 (the Phase 1 default) for the main heatmap

for case_id in case_trades:
    case_grid = grid_df[(grid_df["case_id"] == case_id) & (grid_df["window_pct"] == 0.20)]
    if len(case_grid) == 0:
        continue

    pivot = case_grid.pivot_table(
        values="ks_d", index="lookback", columns="bucket_size", aggfunc="first"
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto",
                   vmin=0, vmax=max(0.3, pivot.values.max()))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Bucket Size")
    ax.set_ylabel("Lookback")

    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                color = "white" if val > 0.15 else "black"
                ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                       fontsize=9, color=color)

    fig.colorbar(im, ax=ax, label="KS D statistic")
    ax.set_title(f"{case_id} — KS D across parameters (window=20%)",
                fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Count how many settings pass D >= 0.15
    passing = (pivot.values >= 0.15).sum()
    total = (~np.isnan(pivot.values)).sum()
    print(f"  {case_id}: {passing}/{total} settings have D >= 0.15")
    print()

In [ ]:
# ── Window percentage sensitivity (at default bucket=200, lookback=10) ──

print("=== Window Percentage Sensitivity (bucket=200, lookback=10) ===")
print()

for case_id in case_trades:
    case_grid = grid_df[
        (grid_df["case_id"] == case_id) &
        (grid_df["bucket_size"] == 200) &
        (grid_df["lookback"] == 10)
    ]
    if len(case_grid) == 0:
        continue

    print(f"{case_id}:")
    for _, row in case_grid.sort_values("window_pct").iterrows():
        sig = "SIG" if row["significant"] else "   "
        print(f"  window={row['window_pct']:.0%}: D={row['ks_d']:.4f} p={row['ks_p']:.4e} {sig}")
    print()

In [ ]:
# ── Robustness assessment ──

print("=== Robustness Assessment ===")
print("Criterion: D >= 0.15 in >= 3 adjacent parameter settings")
print()

robust_cases = []
fragile_cases = []

for case_id in case_trades:
    case_grid = grid_df[grid_df["case_id"] == case_id]
    if len(case_grid) == 0:
        continue

    n_passing = (case_grid["ks_d"] >= 0.15).sum()
    n_total = len(case_grid)
    pct = n_passing / n_total if n_total > 0 else 0

    # Check adjacency: for each passing config, count neighbors that also pass
    passing_configs = case_grid[case_grid["ks_d"] >= 0.15]

    is_robust = n_passing >= 3
    if is_robust:
        robust_cases.append(case_id)
    else:
        fragile_cases.append(case_id)

    status = "ROBUST" if is_robust else "FRAGILE"
    print(f"{case_id}: {n_passing}/{n_total} ({pct:.0%}) pass D >= 0.15 → {status}")

print(f"\nRobust cases: {len(robust_cases)}")
print(f"Fragile cases: {len(fragile_cases)}")

if fragile_cases:
    print(f"\nWARNING: {fragile_cases} are fragile — Phase 1 conclusion may be unreliable for these cases")